In [2]:
!pip install -q google-generativeai pdfplumber

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.4/68.4 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 58.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 68.5 MB/s eta 0:00:00


In [3]:
import json
import pdfplumber
import google.generativeai as genai
import typing_extensions as typing
from google.colab import userdata

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


In [23]:
system_instruction = """
You are an expert HR resume parser.

Follow these rules strictly:
1. Extract ONLY information that is explicitly present in the resume text.
2. If a field is not present, set its value to null. Never guess or invent values.
3. Normalize phone numbers to the format +91XXXXXXXXXX wherever possible.
4. Skills must be a clean list of technology names — no duplicates, proper casing (e.g. Python, React.js, MongoDB).
5. Summary must be exactly one sentence describing the candidate's profile.
6. The resume may be written in English or any Indian language, but all output values must always be in English.
"""

genai.configure(api_key=userdata.get('GEMINI_API'))
model=genai.GenerativeModel(
    "gemini-2.5-flash-lite",
    system_instruction=system_instruction
)

In [7]:
from re import S
class Education(typing.TypedDict):
  degree:str
  institute:str
  year:str
  score:str

class Experience(typing.TypedDict):
  company:str
  role:str
  duration:str
  highlights:list[str]

class Project(typing.TypedDict):
  name:str
  description:str
  tech_stack:list[str]

class Resume(typing.TypedDict):
  name:str
  email:str
  phone:str
  linkedin_url:str
  education:list[Education]
  skills:list[str]
  experience:list[Experience]
  projects:list[Project]
  total_experience_years:float
  summary:str

print(list(Resume.__annotations__.keys()))

['name', 'email', 'phone', 'linkedin_url', 'education', 'skills', 'experience', 'projects', 'total_experience_years', 'summary']


In [22]:
def build_prompt(resume_text: str) -> str:
    return f"""
Parse the following resume and extract all relevant information.
Return a JSON object with exactly these 10 keys:
name, email, phone, linkedin_url, education, skills, experience, projects, total_experience_years, summary.

IMPORTANT: Every key must be present in the output. Use null for missing fields. Never skip a key.

RESUME:
{resume_text}
"""

In [9]:
def strip_markdown_json(text: str)->str:
    """Remove markdown code fences if model wraps output in them."""
    text=text.strip()
    if text.startswith("```"):
        lines=text.splitlines()
        text="\n".join(lines[1:-1]).strip()
    return text


def way1_prompt_only(resume_text:str)->dict:
    prompt = f"""
You are an expert HR resume parser.

Follow these rules strictly:
1. Extract ONLY information explicitly present in the resume. Never invent values.
2. Use null for any missing field.
3. Normalize phone to +91XXXXXXXXXX format.
4. Skills must be a clean list, no duplicates, proper casing.
5. Summary must be exactly one sentence.
6. Output must always be in English even if resume is in another language.

Parse the resume below and return ONLY a valid JSON object with exactly these 10 keys:
name, email, phone, linkedin_url, education, skills, experience, projects, total_experience_years, summary.

Do not add any explanation. Do not wrap in markdown. Just return the JSON object.

RESUME:
{resume_text}
"""

    res=model.generate_content(prompt)
    raw=res.text

    cleaned_data=strip_markdown_json(raw)

    try:
        data=json.loads(cleaned_data)
    except json.JSONDecodeError:
        res=model.generate_content(prompt)
        cleaned_data=strip_markdown_json(res.text)
        data=json.loads(cleaned_data)

    return data


In [10]:
def way2_mime_type(resume_text:str)->dict:
  prompt=build_prompt(resume_text)

  res=model.generate_content(
      prompt,
      generation_config={
          "response_mime_type":"application/json",
          "temperature":0.2,
          "max_output_tokens":2048
      }
  )
  raw=res.text

  try:
    data=json.loads(raw)
  except json.JSONDecodeError:
    res=model.generate_content(
        prompt,
        generation_config={
          "response_mime_type":"application/json",
          "temperature":0.2,
          "max_output_tokens":2048
        }
    )
    data=json.loads(res.text)

  return data


In [24]:
from json.decoder import JSONDecoder
def way3_schema(resume_text:str)->dict:
  prompt=build_prompt(resume_text)

  res=model.generate_content(
      prompt,
      generation_config={
          "response_mime_type":"application/json",
          "response_schema":Resume,
          "temperature":0.15,
          "max_output_tokens":4096
      }
  )
  raw=res.text

  try:
    data=json.loads(raw)
  except json.JSONDecodeError:
    res=model.generate_content(
        prompt,
        generation_config={
          "response_mime_type":"application/json",
          "response_schema":Resume,
          "temperature":0.15,
          "max_output_tokens":4096
      }
    )
    data=json.loads(res.text)

  return data




In [12]:
def run_sanity_checks(data:dict)->dict:
    """
    Run at least 3 sanity checks on parsed data.
    Fix invalid values gracefully instead of crashing.
    """

    email=data.get("email")
    if email and "@" not in email:
      print(f"  [Sanity] Invalid email detected: {email} → setting to null")
      data["email"]=None

    education=data.get("education") or []
    for edu in education:
      year=edu.get("year")
      if year:
        try:
          y=int(year)
          if not (1990<=y<=2026):
            print(f"  [Sanity] Suspicious year detected: {year} → setting to null")
            edu["year"]=None
        except ValueError:
          pass

    exp=data.get("total_experience_years")
    if exp is not None and exp<0:
      print(f"  [Sanity] Negative experience detected: {exp} → setting to 0.0")
      data["total_experience_years"]=0.0

    return data

In [28]:
import time

def print_results(way1: dict, way2: dict, way3: dict, test_label: str):
    """
    Prints full JSON output for all 3 ways + summary line for Way 3.
    """

    print("\n" + "="*60)
    print(f"RESULTS FOR: {test_label}")
    print("="*60)

    print("\n--- WAY 1 (Prompt Only) ---")
    print(json.dumps(way1, indent=2, ensure_ascii=False))

    print("\n--- WAY 2 (MIME Type) ---")
    print(json.dumps(way2, indent=2, ensure_ascii=False))

    print("\n--- WAY 3 (Schema) — Final Validated Output ---")
    print(json.dumps(way3, indent=2, ensure_ascii=False))

    name= way2.get("name") or "N/A"
    email= way2.get("email") or "N/A"
    phone= way2.get("phone") or "N/A"
    skills= way2.get("skills") or []
    exp= way2.get("total_experience_years")

    top_skills = ", ".join(skills[:3]) if skills else "N/A"
    exp_str = f"{exp} years" if exp is not None else "N/A"

    print("\n--- SUMMARY ---")
    print(f"Name            : {name}")
    print(f"Email           : {email}")
    print(f"Phone           : {phone}")
    print(f"Top Skills      : {top_skills}")
    print(f"Total Experience: {exp_str}")
    print("="*60)


def parse_resume(text: str) -> tuple[dict, dict, dict]:
    """
    Runs all 3 Ways. Returns (way1_raw,way2_raw,way3_validated)
    """

    print("\nRunning Way 1 — Prompt Only...")
    result1 = way1_prompt_only(text)
    time.sleep(15)

    print("Running Way 2 — MIME Type...")
    result2 = way2_mime_type(text)
    time.sleep(15)

    print("Running Way 3 — Schema...")
    result3 = way3_schema(text)

    print("Running sanity checks...")
    validated = run_sanity_checks(result3)

    return result1,result2,validated


In [14]:
def extract_text_from_pdf(pdf_path:str)->str:
    """
    Extracts all text from a PDF file page by page.
    Returns a single clean string.
    """

    full_text=[]

    with pdfplumber.open(pdf_path) as pdf:
      for i,page in enumerate(pdf.pages):
        text=page.extract_text()
        if text:
          full_text.append(text)
        else:
          print(f"  [Warning] Page {i+1} returned no text — may be scanned/image-based")

    combined="\n".join(full_text)

    return combined


In [15]:
resume1 = """
ANJALI SHARMA
Email: anjali.sharma@gmail.com | Phone: 9876543210
LinkedIn: linkedin.com/in/anjali-sharma

EDUCATION
- B.Tech in Computer Science, IIT Delhi, 2024 (CGPA: 8.9)
- 12th CBSE, DPS Noida, 2020 (94%)

SKILLS
Python, Django, FastAPI, React.js, MongoDB, PostgreSQL,
Docker, Git, AWS (basics), TensorFlow

EXPERIENCE
Software Engineering Intern, Flipkart (May 2023 - July 2023)
- Built a recommendation microservice using Python + Redis
- Reduced API latency by 35%

Open Source Contributor, scikit-learn (2022 - present)
- Merged 4 PRs related to model serialization

PROJECTS
- AgriBot: WhatsApp chatbot for farmers using Gemini API + Twilio
- StudyMate: Note-summarizer Chrome extension (1.2k users)
"""

print("RUNNING TEST 1 — Anjali Sharma (Well-formatted)")
way1_out, way2_out, way3_out = parse_resume(resume1)
print_results(way1_out, way2_out, way3_out, test_label="Test 1 — Anjali Sharma")


RUNNING TEST 1 — Anjali Sharma (Well-formatted)

Running Way 1 — Prompt Only...
Running Way 2 — MIME Type...
Running Way 3 — Schema...
Running sanity checks...

RESULTS FOR: Test 1 — Anjali Sharma

--- WAY 1 (Prompt Only) ---
{
  "name": "ANJALI SHARMA",
  "email": "anjali.sharma@gmail.com",
  "phone": "+919876543210",
  "linkedin_url": "https://linkedin.com/in/anjali-sharma",
  "education": [
    {
      "degree": "B.Tech in Computer Science",
      "institution": "IIT Delhi",
      "graduation_year": 2024,
      "cgpa": "8.9"
    },
    {
      "degree": "12th CBSE",
      "institution": "DPS Noida",
      "graduation_year": 2020,
      "percentage": "94%"
    }
  ],
  "skills": [
    "Python",
    "Django",
    "FastAPI",
    "React.js",
    "MongoDB",
    "PostgreSQL",
    "Docker",
    "Git",
    "AWS",
    "TensorFlow"
  ],
  "experience": [
    {
      "title": "Software Engineering Intern",
      "company": "Flipkart",
      "start_date": "May 2023",
      "end_date": "July 202

In [25]:
resume2 = """
hii i am ravi (ravi.k@yahoo.in / 8765-432-100). recently finished
my btech from vit vellore (2023, cgpa around 8). i know java/python/c++
and a bit of ml stuff. did an internship at zoho for 6 months as software
trainee. also worked at a startup called bytemate for 8 months as backend
developer where i built their payment gateway. love football and travelling.
also made a small project called NoteShare - a notes sharing app for
college students built with react and firebase.
"""

way1_out, way2_out, way3_out = parse_resume(resume2)
print_results(way1_out, way2_out, way3_out, test_label="Test 2 — Ravi (Messy)")


Running Way 1 — Prompt Only...
Running Way 2 — MIME Type...
Running Way 3 — Schema...
Running sanity checks...

RESULTS FOR: Test 2 — Ravi (Messy)

--- WAY 1 (Prompt Only) ---
{
  "name": "Ravi",
  "email": "ravi.k@yahoo.in",
  "phone": "+918765432100",
  "linkedin_url": null,
  "education": [
    {
      "degree": "B.Tech",
      "institution": "VIT Vellore",
      "year": 2023,
      "cgpa": "8"
    }
  ],
  "skills": [
    "Java",
    "Python",
    "C++",
    "Machine Learning",
    "React",
    "Firebase"
  ],
  "experience": [
    {
      "title": "Backend Developer",
      "company": "Bytemate",
      "duration": "8 months",
      "description": "Built their payment gateway."
    },
    {
      "title": "Software Trainee",
      "company": "Zoho",
      "duration": "6 months",
      "description": null
    }
  ],
  "projects": [
    {
      "name": "NoteShare",
      "description": "A notes sharing app for college students built with React and Firebase."
    }
  ],
  "total_expe

In [19]:
pdf_path = "/content/shivam_resume.pdf"
resume3 = extract_text_from_pdf(pdf_path)

print("Extracted text preview:")
print(resume3[:500])

Extracted text preview:
Shivam Petkar
Surat, Gujarat
(cid:131) +91 7069329209 | shivampetkar2@gmail.com | (cid:239) LinkedIn | § GitHub
Education
Institute of Technology, Nirma University July 2023 – June 2027
B.Tech in Computer Science and Engineering | CGPA: 8.41/10 Ahmedabad, Gujarat
Technical Skills
Languages: C++, Java, Python, HTML, CSS, JavaScript
Frameworks/Libraries: React.js, Next.js, Node.js, Express.js, FastAPI
Platform: kubernetes, GCP
Tools: Git, GitHub, docker
Databases: SQLite, SQLAlchemy ORM, MySQL, Or


In [27]:
way1_out, way2_out, way3_out = parse_resume(resume3)
print_results(way1_out, way2_out, way3_out, test_label="Test 3 — PDF Resume")


Running Way 1 — Prompt Only...
Running Way 2 — MIME Type...
Running Way 3 — Schema...
Running sanity checks...

RESULTS FOR: Test 3 — PDF Resume

--- WAY 1 (Prompt Only) ---
{
  "name": "Shivam Petkar",
  "email": "shivampetkar2@gmail.com",
  "phone": "+917069329209",
  "linkedin_url": null,
  "education": [
    {
      "degree": "B.Tech in Computer Science and Engineering",
      "university": "Institute of Technology, Nirma University",
      "start_date": "July 2023",
      "end_date": "June 2027",
      "cgpa": "8.41/10"
    }
  ],
  "skills": [
    "C++",
    "Java",
    "Python",
    "HTML",
    "CSS",
    "JavaScript",
    "React.js",
    "Next.js",
    "Node.js",
    "Express.js",
    "FastAPI",
    "Kubernetes",
    "GCP",
    "Git",
    "GitHub",
    "Docker",
    "SQLite",
    "SQLAlchemy ORM",
    "MySQL",
    "Oracle",
    "MongoDB",
    "PostgreSQL",
    "Typescript",
    "Tailwind",
    "Socket.io",
    "Redis",
    "XGBoost",
    "Scikit-learn",
    "Streamlit"
  ],
  

Note on Way 3 (Schema-based) Output Quality
The schema-based approach (Way 3) is theoretically the most reliable method — it enforces exact keys, exact types, and structure at the API level. However, in this implementation the output appears incomplete compared to Way 1 and Way 2 due to the following constraints:

Model limitation — gemini-2.5-flash-lite (as required by the assignment) has known limitations with complex nested TypedDict schemas. It tends to drop fields it is uncertain about rather than explicitly returning null, unlike larger models like gemini-1.5-flash or gemini-2.5-flash which handle nested schemas reliably.
Free tier quota — Rate limiting (20 requests/minute) forced the use of time.sleep() delays and occasional retries, which affected output consistency across runs.
Schema strictness vs null handling — When a response_schema is passed, the model prioritizes type correctness over completeness. Fields it cannot confidently populate are silently dropped instead of being set to null, which is the opposite behavior compared to prompt-only and MIME-type approaches where null handling is explicitly instructed in plain English.